# NPDES Data Cleaning: ECHO Industry Codes (NAICS & SIC, Iowa)

Cleans the two ECHO industry-classification crosswalks for Iowa NPDES
facilities — one for **NAICS** codes, one for the legacy **SIC** codes — into
parallel tidy tables. They share an identical shape (permit → code → description
→ primary flag), so they are cleaned together here and written to two files.

**Inputs:**
  - `data/tabular/01_raw/npdes/echo-naics-iowa.csv`
  - `data/tabular/01_raw/npdes/echo-sics-iowa.csv`

**Outputs:**
  - `data/tabular/02_clean/npdes/echo-naics-clean.csv`
  - `data/tabular/02_clean/npdes/echo-sics-clean.csv`

Each row links one facility (`npdes_id`) to one industry `code`/`description`,
with `is_primary` marking the facility's primary classification.

In [1]:
import pandas as pd
from pathlib import Path


def find_repo_root(start: Path | None = None) -> Path:
    """Walk upward until we find the repo's data/tabular directory.

    Notebooks have no __file__, and the kernel's working directory varies, so
    resolving paths relative to a fixed number of "../" is fragile. Searching
    upward for a sentinel makes the notebook runnable from anywhere.
    """
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / "data" / "tabular").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repo root containing data/tabular/")


REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / "data" / "tabular" / "01_raw" / "npdes"
CLEAN_DIR = REPO_ROOT / "data" / "tabular" / "02_clean" / "npdes"
CLEAN_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO_ROOT)
print("Raw dir:  ", RAW_DIR)
print("Clean dir:", CLEAN_DIR)

Repo root: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction
Raw dir:   /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/01_raw/npdes
Clean dir: /Users/owenm/Documents/Owens_files/College Stuff/Research Projects/STEEN Fellowship Project 2026/workspace/Water-Quality-Prediction/data/tabular/02_clean/npdes


## Step 1 — A shared cleaner

Both files have columns `NPDES_ID`, `<SYS>_CODE`, `<SYS>_DESC`,
`PRIMARY_INDICATOR_FLAG`. The cleaner reads codes as strings (they are
categorical identifiers, not quantities), lower-cases the column names, strips
whitespace, converts the `Y`/`N` flag to a boolean `is_primary`, drops exact
duplicates, and confirms `(npdes_id, code)` is unique.

In [2]:
def clean_codes(filename: str, system: str) -> pd.DataFrame:
    df = pd.read_csv(RAW_DIR / filename, dtype="string")
    code_col, desc_col = f"{system}_CODE", f"{system}_DESC"

    out = pd.DataFrame({
        "npdes_id": df["NPDES_ID"].str.strip(),
        "code": df[code_col].str.strip(),
        "description": df[desc_col].str.strip(),
        "is_primary": df["PRIMARY_INDICATOR_FLAG"].str.strip().eq("Y"),
    })

    flags = set(df["PRIMARY_INDICATOR_FLAG"].dropna().unique())
    assert flags <= {"Y", "N"}, f"unexpected primary flags: {flags}"

    n_raw = len(out)
    out = out.drop_duplicates().sort_values(["npdes_id", "code"]).reset_index(drop=True)
    dupes = out.duplicated(["npdes_id", "code"]).sum()
    assert dupes == 0, f"{dupes} duplicate (npdes_id, code) rows in {filename}"
    print(f"{filename}: {n_raw:,} raw rows -> {len(out):,} clean "
          f"({out['npdes_id'].nunique()} facilities, {out['code'].nunique()} codes)")
    return out

## Step 2 — Clean NAICS

In [3]:
naics = clean_codes("echo-naics-iowa.csv", "NAICS")
naics.head()

echo-naics-iowa.csv: 1,668 raw rows -> 1,668 clean (1668 facilities, 154 codes)


,npdes_id,code,description,is_primary
0,IA0000035,221310,Water Supply and Irrigation Systems,True
1,IA0000051,333111,Farm Machinery and Equipment Manufacturing,True
2,IA0000060,333111,Farm Machinery and Equipment Manufacturing,True
3,IA0000108,221112,Fossil Fuel Electric Power Generation,True
4,IA0000132,221112,Fossil Fuel Electric Power Generation,True


## Step 3 — Clean SIC

In [4]:
sics = clean_codes("echo-sics-iowa.csv", "SIC")
sics.head()

echo-sics-iowa.csv: 1,693 raw rows -> 1,693 clean (1692 facilities, 152 codes)


,npdes_id,code,description,is_primary
0,IA0000035,4941,Water Supply,True
1,IA0000051,3523,Farm Machinery And Equipment,True
2,IA0000060,3523,Farm Machinery And Equipment,True
3,IA0000108,4911,Electric Services,True
4,IA0000132,4911,Electric Services,True


## Step 4 — Write both tables

In [5]:
for df_, fname in [(naics, "echo-naics-clean.csv"), (sics, "echo-sics-clean.csv")]:
    out_path = CLEAN_DIR / fname
    df_.to_csv(out_path, index=False)
    print(f"Wrote {len(df_):,} rows × {df_.shape[1]} cols to {out_path.relative_to(REPO_ROOT)}")

Wrote 1,668 rows × 4 cols to data/tabular/02_clean/npdes/echo-naics-clean.csv
Wrote 1,693 rows × 4 cols to data/tabular/02_clean/npdes/echo-sics-clean.csv
